from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# ---------- SETUP DRIVER ----------
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

# ---------- URL SẢN PHẨM ----------
url = "https://cellphones.com.vn/laptop-lenovo-ideapad-slim-3-14irh10-83k00008vn.html"

driver.get(url)
time.sleep(2)   # để trang khởi động JS

# ---------- 1. SCROLL ĐẾN KHỐI THÔNG SỐ ----------
try:
    tech_block = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.cps-block-technicalInfo"))
    )

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", tech_block)
    time.sleep(1.2)  # ĐỂ TRIGGER render lazy
except Exception as e:
    print("Không tìm thấy block kỹ thuật:", e)

# ---------- 2. CHỜ NÚT XEM TẤT CẢ XUẤT HIỆN SAU KHI RENDER ----------
try:
    show_btn = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.XPATH, "//button[contains(., 'Xem tất cả')]"))
    )

    # Scroll nút vào giữa cho chắc
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", show_btn)
    time.sleep(0.3)

    # CLICK BẰNG JS (TRÁNH CHE KHUẤT, INTERCEPT)
    driver.execute_script("arguments[0].click();", show_btn)

    print(">>> ĐÃ CLICK NÚT XEM TẤT CẢ THÀNH CÔNG!")

except Exception as e:
    print("Không click được nút:", e)



In [15]:
import json
import os
import re
from bs4 import BeautifulSoup
from pprint import pprint
from pathlib import Path

# ---------- Config ----------
# 🌟 CHÚ Ý: Đổi đường dẫn này cho đúng với vị trí file của bạn
MANIFEST_PATH = Path(r"D:\DS_project\DS_AI-Laptop-Advisor-System\data\cellphones\raw_htmls_manifest.json")
TEST_COUNT = 10# chỉnh số mẫu muốn test

# 🌟 CẦN THIẾT: Đường dẫn thư mục gốc của dự án để ghép với đường dẫn tương đối trong JSON
# Thay đổi nếu cấu trúc thư mục của bạn khác
PROJECT_ROOT = MANIFEST_PATH.parent.parent.parent 


# ---------- Helpers ----------
def extract_name_from_url(url: str) -> str:
    # Đã sửa lỗi NoneType và SyntaxWarning
    if not isinstance(url, str) or not url:
        return "" 
    
    slug = url.rstrip("/").split("/")[-1]
    if slug.endswith(".html"):
        slug = slug[:-5]
    name = slug.replace("-", " ")
    name = re.sub(r"\s+", " ", name).strip() 
    name = " ".join([w.upper() if w.lower() in ("hp","lenovo","dell","acer","asus","msi","apple","macbook","intel","amd") else w.capitalize() for w in name.split()])
    return name

def clean_product_name(name:str)->str:
    parts = name.split()
    if parts and parts[0].lower()=="laptop":
        parts = parts[1:]
    return " ".join(parts)

def load_file_text(path):
    # Đã sửa lỗi NoneType
    if not isinstance(path, (str, Path)) or not path:
        return None
        
    path = os.path.normpath(path)
    
    if not os.path.exists(path):
        return None
        
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

# ---------- Parse raw specs table -> dict label->value (Giữ nguyên) ----------
def parse_specs_html(detail_html: str) -> dict:
    if not detail_html:
        return {}
    soup = BeautifulSoup(detail_html, "html.parser")
    specs = {}
    rows = soup.select("table.technical-content tr.technical-content-item")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 2:
            k = cols[0].get_text(" ", strip=True)
            v = cols[1].get_text(" ", strip=True)
            specs[k] = v
    # fallback: generic tr/td pairs
    if not specs:
        for r in soup.select("tr"):
            cols = r.find_all("td")
            if len(cols) >= 2:
                k = cols[0].get_text(" ", strip=True)
                v = cols[1].get_text(" ", strip=True)
                if k:
                    specs[k] = v
    return specs

# ---------- Field parsers ----------
def parse_cpu(cpu_raw: str):
    if not cpu_raw: return (None, None, None, None)
    cpu = cpu_raw.lower()
    cpu_manufacturer = "Intel" if "intel" in cpu else ("AMD" if "amd" in cpu else None)
    m = re.search(r"(i[3579]|ryzen\s*\d+)", cpu, re.I)
    brand_mod = m.group(1).title() if m else None
    gen = None
    m2 = re.search(r"([0-9]{3,5})", cpu.replace("-", ""))
    if m2:
        num = m2.group(1)
        if cpu_manufacturer == "Intel" and len(num) >= 4:
            gen = num[:2]
        elif cpu_manufacturer == "AMD":
            gen = num[0]
    speed = None
    m3 = re.search(r"(\d+(?:\.\d+)?)\s*ghz", cpu, re.I)
    if m3:
        try:
            speed = float(m3.group(1))
        except:
            speed = None
    return cpu_manufacturer, brand_mod, gen, speed

# 🌟 HÀM ĐƯỢC SỬA: Xử lý dấu gạch ngang cho Bus Speed
def parse_ram(ram_raw: str, ram_type_raw: str):
    ram_gb = None
    if ram_raw:
        m = re.search(r"(\d+)\s*gb", ram_raw, re.I)
        if m:
            ram_gb = int(m.group(1))
            
    ram_type = None
    bus = None
    if ram_type_raw:
        # 🌟 KHẮC PHỤC DẤU GẠCH NGANG: Thay thế dấu gạch ngang bằng khoảng trắng để dễ dàng bắt số.
        normalized_raw = ram_type_raw.replace('-', ' ') 

        # 1. Bắt Loại RAM (DDRx)
        m = re.search(r"(ddr\d)", normalized_raw, re.I)
        if m:
            ram_type = m.group(1).upper()
            
        # 2. Bắt Bus Speed:
        # Cố gắng bắt Bus + đơn vị (mhz|mt/s)
        m2 = re.search(r"(\d{3,4})\s*(mhz|mt/s)", normalized_raw, re.I)
        if m2:
            bus = int(m2.group(1))
        else:
            # Nếu không tìm thấy đơn vị, tìm số 3-4 chữ số bất kỳ (Bus speed thường >= 1000)
            m3 = re.search(r"(\d{3,4})", normalized_raw, re.I)
            if m3:
                num_candidate = int(m3.group(1))
                if num_candidate >= 1000:
                    bus = num_candidate
            
    return ram_gb, ram_type, bus

def parse_storage(raw: str):
    if not raw: return None
    m = re.search(r"(\d+)\s*tb", raw, re.I)
    if m:
        return int(m.group(1))*1024
    m = re.search(r"(\d+)\s*gb", raw, re.I)
    if m:
        return int(m.group(1))
    return None

def parse_screen(size_raw, res_raw):
    size = None
    if size_raw:
        m = re.search(r"(\d+(?:\.\d+)?)\s*(inches|inch|in)?", size_raw, re.I)
        if m:
            size = float(m.group(1))
    resolution = None
    
    if res_raw:
        m = re.search(r"(\d{3,4}\s*[x×]\s*\d{3,4})", res_raw.replace(" ", ""))
        if m:
            resolution = m.group(1).replace("×","x")
            
    # Tần số quét đã được tách ra khỏi đây và xử lý riêng biệt trong normalize_specs
    refresh = None 
    return size, resolution, refresh # refresh luôn là None ở đây, được xử lý trong normalize_specs

# 🌟 HÀM MỚI: Xử lý Tần số quét (Refresh Rate) độc lập
def parse_refresh_rate(raw: str):
    if not raw: return None
    m = re.search(r"(\d{2,3})\s*hz", raw, re.I)
    if m:
        return int(m.group(1))
    return None

def parse_gpu(raw: str):
    if not raw: return None
    r = raw.lower()
    if "nvidia" in r or "geforce" in r:
        return "NVIDIA"
    if "intel" in r:
        return "Intel"
    if "amd" in r or "radeon" in r:
        return "AMD"
    return None

def parse_weight(raw: str):
    if not raw: return None
    m = re.search(r"(\d+(?:\.\d+)?)\s*kg", raw, re.I)
    if m:
        return float(m.group(1))
    m2 = re.search(r"(\d+)\s*g", raw, re.I)
    if m2:
        return float(m2.group(1))/1000.0
    return None

def parse_battery(raw: str):
    return raw.strip() if raw else None

def parse_price_to_int(price_raw: str):
    if not price_raw:
        return None
    nums = re.sub(r"[^\d]", "", price_raw)
    return int(nums) if nums else None

# ---------- Normalize into requested fields ----------
def normalize_specs(product_name, price_raw, specs_html, manifest):
    specs_map = parse_specs_html(specs_html)

    cpu_raw = specs_map.get("Loại CPU") or specs_map.get("Bộ xử lý") or specs_map.get("Loại bộ vi xử lý") or specs_map.get("CPU") or ""
    gpu_raw = specs_map.get("Loại card đồ họa") or specs_map.get("Đồ họa") or specs_map.get("Card đồ họa") or specs_map.get("GPU") or ""
    ram_raw = specs_map.get("Dung lượng RAM") or specs_map.get("RAM") or specs_map.get("Bộ nhớ RAM") or ""
    ram_type_raw = specs_map.get("Loại RAM") or specs_map.get("Kiểu RAM") or specs_map.get("Ram Type") or ""
    storage_raw = specs_map.get("Ổ cứng") or specs_map.get("Bộ nhớ trong") or specs_map.get("Lưu trữ") or specs_map.get("Ổ lưu trữ") or ""
    screen_size_raw = specs_map.get("Kích thước màn hình") or specs_map.get("Kích thước") or specs_map.get("Màn hình") or ""
    screen_res_raw = specs_map.get("Độ phân giải màn hình") or specs_map.get("Độ phân giải") or ""
    weight_raw = specs_map.get("Trọng lượng") or specs_map.get("Cân nặng") or ""
    battery_raw = specs_map.get("Pin") or specs_map.get("Dung lượng pin") or ""
    
    # Lấy Tần số quét trực tiếp từ key "Tần số quét"
    refresh_raw = specs_map.get("Tần số quét") or ""

    cpu_manufacturer, cpu_brand, cpu_gen, cpu_speed = parse_cpu(cpu_raw)
    ram_gb, ram_type, bus = parse_ram(ram_raw, ram_type_raw)
    storage_gb = parse_storage(storage_raw)
    
    # Tần số quét được xử lý riêng
    screen_size, resolution, _ = parse_screen(screen_size_raw, screen_res_raw)
    refresh_rate = parse_refresh_rate(refresh_raw) # Sử dụng hàm parse_refresh_rate

    gpu_manu = parse_gpu(gpu_raw)
    weight_kg = parse_weight(weight_raw)
    battery = parse_battery(battery_raw)
    price_vnd = parse_price_to_int(price_raw)

    manufacturer = manifest.get("manufacturer") or (product_name.split()[0].lower() if product_name else None)

    return {
        "Product Name": product_name,
        "Manufacturer": manufacturer,
        "CPU manufacturer": cpu_manufacturer,
        "CPU brand modifier": cpu_brand,
        "CPU generation": cpu_gen,
        "CPU Speed (GHz)": cpu_speed,
        "RAM (GB)": ram_gb,
        "RAM Type": ram_type,
        "Bus (MHz)": bus,
        "Storage (GB)": storage_gb,
        "Screen Size (inch)": screen_size,
        "Screen Resolution": resolution,
        "Refresh Rate (Hz)": refresh_rate,
        "GPU manufacturer": gpu_manu,
        "Weight (kg)": weight_kg,
        "Battery": battery,
        "Price (VND)": price_vnd,
        "url": manifest.get("url"),
        "saved_path": manifest.get("saved_path"),
        "detail_specs_html_path": manifest.get("detail_specs_html_path")
    }

# ---------- Load manifest and test ----------
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifests = json.load(f)

print(f"Bắt đầu xử lý {TEST_COUNT} mẫu từ {MANIFEST_PATH}")
for i, manifest in enumerate(manifests[:TEST_COUNT]):
    print("\n==========================")
    print("INDEX:", i)
    url = manifest.get("url")
    print("URL:", url)
    print(type(url))
    
    # Product name from URL cleaned
    product_name = clean_product_name(extract_name_from_url(url))
    print("Product Name (from url):", product_name)
    
    # load specs html
    detail_relative_path = manifest.get("detail_specs_html_path")
    
    # Ghép đường dẫn tương đối với thư mục gốc
    if detail_relative_path:
        detail_path = os.path.join(PROJECT_ROOT, detail_relative_path)
    else:
        detail_path = None
        
    detail_html = load_file_text(detail_path)
    
    if detail_html is None:
        print(f"❌ detail_specs_html_path not found: {detail_path}")
        continue

    # Xử lý thành công
    result = normalize_specs(product_name, manifest.get("price"), detail_html, manifest)
    print(result)

Bắt đầu xử lý 10 mẫu từ D:\DS_project\DS_AI-Laptop-Advisor-System\data\cellphones\raw_htmls_manifest.json

INDEX: 0
URL: https://cellphones.com.vn/laptop-hp-245-g10-b8pf8at.html
<class 'str'>
Product Name (from url): HP 245 G10 B8pf8at
{'Product Name': 'HP 245 G10 B8pf8at', 'Manufacturer': 'hp', 'CPU manufacturer': 'AMD', 'CPU brand modifier': 'Ryzen 5', 'CPU generation': '7', 'CPU Speed (GHz)': 4.5, 'RAM (GB)': 16, 'RAM Type': 'DDR4', 'Bus (MHz)': 3200, 'Storage (GB)': 256, 'Screen Size (inch)': 14.0, 'Screen Resolution': '1920x1080', 'Refresh Rate (Hz)': None, 'GPU manufacturer': 'AMD', 'Weight (kg)': 1.36, 'Battery': 'HP Long Life 3-cell, 41 Wh Li-ion', 'Price (VND)': 15590000, 'url': 'https://cellphones.com.vn/laptop-hp-245-g10-b8pf8at.html', 'saved_path': 'data/cellphones/raw_htmls\\hp_2.html', 'detail_specs_html_path': 'data/cellphones/detail_htmls\\hp_2_specs.html'}

INDEX: 1
URL: https://cellphones.com.vn/laptop-hp-victus-15-fa2731tx-b85lnpa.html
<class 'str'>
Product Name (fro